In [1]:
import os
import json
import copy
import math
import random
import warnings
from datetime import datetime, timezone
from pathlib import Path
import hashlib
from typing import Dict, List, Optional, Tuple

from joblib import dump, load

import textwrap
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from IPython.display import display

from sklearn.decomposition import PCA as CPU_PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Reproducibility
BASE_SEED = 19537
ALL_SEEDS = [19537, 1584678, 17052356]

SEED = BASE_SEED
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

RNG = np.random.default_rng(SEED)

def set_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    RNG = np.random.default_rng(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {TORCH_DEVICE}")


Torch device: cuda


In [2]:
# Paths
ARTIFACTS = Path("artifacts")
IN_CLEAN  = ARTIFACTS / "cleaned"  / "notebook 2"
IN_T2     = ARTIFACTS / "aligned"  / "notebook 1" / "track2_nonintersection"
IN_META   = ARTIFACTS / "metadata"

NOTEBOOK_SUBDIR = "notebook 6_remasker_prot"
OUT_REPORTS = ARTIFACTS / "reports"  / NOTEBOOK_SUBDIR
OUT_META    = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR
OUT_CACHE   = ARTIFACTS / "cache"    / NOTEBOOK_SUBDIR

for d in [OUT_REPORTS, OUT_META, OUT_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

REMASKER_LOCK_FILENAME = "remasker_imputer_choice_prot.json"


In [3]:
# Backbone lock
LOCK_PATH = IN_META / "proteomics_backbone_lock.json"
if not LOCK_PATH.exists():
    raise FileNotFoundError(
        f"proteomics_backbone_lock.json not found at {LOCK_PATH}. "
        "Run the lock cell at the end of Notebook 3b first."
    )

with LOCK_PATH.open("r", encoding="utf-8") as f:
    backbone_lock = json.load(f)

PRIMARY_ARM   = backbone_lock["track1_primary_arm"]
SECONDARY_ARM = backbone_lock["track1_secondary_arm"]
TRACK2_ARM    = backbone_lock["track2_stress_test_arm"]

print("Backbone lock loaded.")
print(f"  Primary arm  : {PRIMARY_ARM}")
print(f"  Secondary arm: {SECONDARY_ARM}")
print(f"  Track 2 arm  : {TRACK2_ARM}")
print(f"  Seeds to run : {ALL_SEEDS}")


Backbone lock loaded.
  Primary arm  : prot_procan_depmapSanger
  Secondary arm: prot_ms_ccle_gygi
  Track 2 arm  : prot_combined_union
  Seeds to run : [19537, 1584678, 17052356]


In [4]:
# Full feature-combination grid for ReMasker proteomics bake-off
FULL_FEATURE_SETS = [
    "rna",
    "cnv",
    "mut",
    "prot",
    "rna+cnv",
    "rna+mut",
    "rna+prot",
    "cnv+mut",
    "cnv+prot",
    "mut+prot",
    "rna+cnv+mut",
    "rna+cnv+prot",
    "rna+mut+prot",
    "cnv+mut+prot",
    "rna+cnv+mut+prot",
]

# Same proteomics arms as before
ACTIVE_ARMS = [
    "prot_combined_union",
    "prot_procan_depmapSanger",
    "prot_rppa_ccle",
    "prot_ms_ccle_gygi",
]

# This notebook currently supports Ridge and ElasticNet.
# If you only want ElasticNet, set ACTIVE_MODELS = ["elasticnet"] instead.
ACTIVE_MODELS = ["elasticnet", "ridge"]

REMASKER_TEST_CONFIGS = [
    {"rank": rank, "arm": arm, "model": model, "feature_set": fs}
    for rank, (arm, model, fs) in enumerate(
        [
            (arm, model, fs)
            for arm in ACTIVE_ARMS
            for model in ACTIVE_MODELS
            for fs in FULL_FEATURE_SETS
        ],
        start=1,
    )
]

CONFIGS_BY_ARM = {
    arm: [cfg for cfg in REMASKER_TEST_CONFIGS if cfg["arm"] == arm]
    for arm in ACTIVE_ARMS
}

EXPECTED_CONFIG_KEYS = {
    (int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"]))
    for cfg in REMASKER_TEST_CONFIGS
}

EXPECTED_CONFIG_KEYS_BY_ARM = {
    arm: {
        (int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"]))
        for cfg in cfgs
    }
    for arm, cfgs in CONFIGS_BY_ARM.items()
}

def checkpoint_matches_expected_grid(df: pd.DataFrame, expected_keys) -> bool:
    needed = {"config_rank", "arm", "model", "feature_set"}
    if not needed.issubset(df.columns):
        return False

    found = {
        (int(r["config_rank"]), str(r["arm"]), str(r["model"]).lower(), str(r["feature_set"]))
        for _, r in df[["config_rank", "arm", "model", "feature_set"]]
        .drop_duplicates()
        .iterrows()
    }
    return expected_keys.issubset(found)

print("\nFull ReMasker configuration grid loaded.")
print("Active arms:", ACTIVE_ARMS)
print("Active models:", ACTIVE_MODELS)
print("Feature sets:", FULL_FEATURE_SETS)

display(pd.DataFrame(REMASKER_TEST_CONFIGS))



Full ReMasker configuration grid loaded.
Active arms: ['prot_combined_union', 'prot_procan_depmapSanger', 'prot_rppa_ccle', 'prot_ms_ccle_gygi']
Active models: ['elasticnet', 'ridge']
Feature sets: ['rna', 'cnv', 'mut', 'prot', 'rna+cnv', 'rna+mut', 'rna+prot', 'cnv+mut', 'cnv+prot', 'mut+prot', 'rna+cnv+mut', 'rna+cnv+prot', 'rna+mut+prot', 'cnv+mut+prot', 'rna+cnv+mut+prot']


,rank,arm,model,feature_set
0,1,prot_combined_union,elasticnet,rna
1,2,prot_combined_union,elasticnet,cnv
2,3,prot_combined_union,elasticnet,mut
3,4,prot_combined_union,elasticnet,prot
4,5,prot_combined_union,elasticnet,rna+cnv
...,...,...,...,...
115,116,prot_ms_ccle_gygi,ridge,rna+cnv+mut
116,117,prot_ms_ccle_gygi,ridge,rna+cnv+prot
117,118,prot_ms_ccle_gygi,ridge,rna+mut+prot
118,119,prot_ms_ccle_gygi,ridge,cnv+mut+prot


In [5]:
# Settings
N_DRUGS_BAKEOFF    = 100
MIN_CELLS_PER_DRUG = 80
N_SPLITS_DESIRED   = 10
RIDGE_ALPHA        = 1.0
EN_ALPHA           = 0.05
EN_L1_RATIO        = 0.2
PCA_COMPONENTS     = 100
PRIMARY_TARGET     = "auc"

# ReMasker-style settings
PATCH_SIZE             = 128
REMASK_RATIO           = 0.30
REMASKER_EMBED_DIM     = 64
REMASKER_DEPTH         = 4
REMASKER_NUM_HEADS     = 4
REMASKER_FF_MULT       = 2.0
REMASKER_DROPOUT       = 0.10
REMASKER_EPOCHS        = 120
REMASKER_PATIENCE      = 15
REMASKER_BATCH_SIZE    = 32
REMASKER_LR            = 1e-3
REMASKER_WEIGHT_DECAY  = 1e-5
VAL_FRAC               = 0.15
MIN_VAL_SAMPLES        = 24

USE_AUX_FOR_PROT_IMPUTATION = True
AUX_FOR_PROT_NAME = "aux_rna_cnv_mut"

CACHE_VERSION = "v1_remasker_prot"
CACHE_TAG = (
    f"{CACHE_VERSION}"
    f"__pca{PCA_COMPONENTS}"
    f"__patch{PATCH_SIZE}"
    f"__mask{int(REMASK_RATIO * 100)}"
    f"__ep{REMASKER_EPOCHS}"
)


In [6]:
# Helpers
def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)

def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df

def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])

def safe_nanmean(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.mean()) if x.size > 0 else np.nan

def safe_nanmedian(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if x.size > 0 else np.nan

def safe_nanstd(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.std()) if x.size > 0 else np.nan

def pick_group_column(cell_index: pd.DataFrame) -> str:
    for c in ["lineage_1", "primary_disease", "lineage", "lineage_2"]:
        if c in cell_index.columns:
            return c
    return "depmap_id"

def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int,
    seed: int,
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    groups = groups.reindex(cells).fillna("Unknown").astype(str)
    n_groups = groups.nunique()
    n_cells = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)
    if n_splits >= 2 and n_groups >= 2:
        sp = GroupKFold(n_splits=n_splits)
        return (
            list(sp.split(np.zeros((n_cells, 1)), np.zeros(n_cells), groups.values)),
            f"GroupKFold(n_splits={n_splits})",
        )
    n_splits = min(max(2, n_splits), n_cells)
    sp = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(sp.split(np.zeros((n_cells, 1)))), f"KFold(n_splits={n_splits})"

def parse_feature_set(feature_set: str) -> Tuple[str, ...]:
    if feature_set is None:
        return tuple()
    feature_set = str(feature_set).strip()
    if feature_set == "":
        return tuple()
    return tuple(feature_set.split("+"))

def concat_selected_modalities(
    mats: Dict[str, np.ndarray],
    selected_keys: Tuple[str, ...],
    n_rows: int,
) -> np.ndarray:
    parts = []
    for k in selected_keys:
        if k == "prot":
            continue
        arr = mats.get(k)
        if arr is None or arr.shape[1] == 0:
            return np.zeros((n_rows, 0), dtype=np.float32)
        parts.append(arr)
    if not parts:
        return np.zeros((n_rows, 0), dtype=np.float32)
    return np.concatenate(parts, axis=1)

def make_model(model_name: str, seed: int):
    model_name = str(model_name).lower()

    if model_name == "ridge":
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            Ridge(
                alpha=RIDGE_ALPHA,
                solver="lsqr",
            ),
        )

    if model_name == "elasticnet":
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            ElasticNet(
                alpha=EN_ALPHA,
                l1_ratio=EN_L1_RATIO,
                random_state=seed,
                max_iter=10000,
            ),
        )

    raise ValueError(f"Unsupported model for Notebook 6 ReMasker bake-off: {model_name}")

def _safe_name(s: str) -> str:
    s = str(s)
    return (
        s.replace(os.sep, "_")
         .replace(" ", "_")
         .replace("+", "plus")
         .replace(":", "_")
         .replace("/", "_")
         .replace("\\", "_")
    )


def _cache_digest(**parts) -> str:
    payload = json.dumps(
        parts,
        sort_keys=True,
        default=str,
        separators=(",", ":"),
    )
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:20]


def _cache_file(dirpath: Path, prefix: str, **parts) -> Path:
    dirpath.mkdir(parents=True, exist_ok=True)
    digest = _cache_digest(**parts)
    return dirpath / f"{prefix}__{digest}.joblib"


def _eval_cache_dir(kind: str) -> Path:
    path = OUT_CACHE / kind
    path.mkdir(parents=True, exist_ok=True)
    return path


def _eval_cache_path(
    kind: str,
    seed: int,
    arm: str,
    drug: str,
    fold_i: int,
    cfg_rank: int,
    model_name: str,
    feature_set: str,
    imputer_strategy: str,
) -> Path:
    return _cache_file(
        _eval_cache_dir(kind),
        "eval",
        cache_tag=CACHE_TAG,
        kind=kind,
        seed=seed,
        arm=arm,
        drug=drug,
        fold_i=fold_i,
        cfg_rank=cfg_rank,
        model_name=model_name,
        feature_set=feature_set,
        imputer_strategy=imputer_strategy,
    )


def load_or_run_eval_cache(
    *,
    kind: str,
    seed: int,
    arm: str,
    drug: str,
    fold_i: int,
    cfg_rank: int,
    model_name: str,
    feature_set: str,
    imputer_strategy: str,
    extra_meta: dict,
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    path = _eval_cache_path(
        kind=kind,
        seed=seed,
        arm=arm,
        drug=drug,
        fold_i=fold_i,
        cfg_rank=cfg_rank,
        model_name=model_name,
        feature_set=feature_set,
        imputer_strategy=imputer_strategy,
    )

    if path.exists():
        return load(path)

    mdl = make_model(model_name, seed)
    mdl.fit(X_train, y_train)
    pred = mdl.predict(X_test)

    row = {
        "seed": seed,
        "config_rank": cfg_rank,
        "arm": arm,
        "model": model_name,
        "feature_set": feature_set,
        "uses_prot": extra_meta.get("uses_prot"),
        "compound_id": drug,
        "fold": fold_i,
        "imputer_strategy": imputer_strategy,
        "add_indicators": extra_meta.get("add_indicators", False),
        "force_indicators": extra_meta.get("force_indicators", False),
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "spearman": spearman_corr(y_test, pred),
        "r2": float(r2_score(y_test, pred)),
    }

    dump(row, path)
    return row

In [7]:
# Base fold-safe transforms for backbone modalities
class BaseModalityTransformer:
    def __init__(self, n_components: int, random_state: int):
        self.n_components = n_components
        self.random_state = random_state
        self._imp    = SimpleImputer(strategy="median")
        self._scaler = StandardScaler()
        self._pca    = None
        self._keep   = None

    def fit(self, X: np.ndarray) -> "BaseModalityTransformer":
        X = np.asarray(X, dtype=float)
        self._keep = np.isfinite(X).any(axis=0)
        if self._keep.sum() == 0:
            return self
        Xk = X[:, self._keep]
        Xs = self._scaler.fit_transform(self._imp.fit_transform(Xk))
        n, d = Xs.shape
        n_comp = min(self.n_components, max(1, n - 1), d)
        self._pca = CPU_PCA(n_components=n_comp, random_state=self.random_state)
        self._pca.fit(Xs)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if self._keep is None or self._keep.sum() == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        Xk = X[:, self._keep]
        Xs = self._scaler.transform(self._imp.transform(Xk))
        if self._pca is not None:
            Xs = self._pca.transform(Xs)
        return Xs.astype(np.float32)

def _base_cache_path(seed: int, arm: str, fold_i: int) -> Path:
    return OUT_CACHE / (
        f"base__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}.joblib"
    )

def load_or_build_base_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    train_cells: List[str],
    eligible_cells: List[str],
    rna_df: pd.DataFrame,
    cnv_df: pd.DataFrame,
    mut_df: pd.DataFrame,
) -> dict:
    path = _base_cache_path(seed, arm, fold_i)
    if path.exists():
        return load(path)

    rna_tr = BaseModalityTransformer(PCA_COMPONENTS, seed + 0).fit(
        rna_df.loc[train_cells].to_numpy(dtype=float)
    )
    cnv_tr = BaseModalityTransformer(PCA_COMPONENTS, seed + 1).fit(
        cnv_df.loc[train_cells].to_numpy(dtype=float)
    )
    mut_tr = BaseModalityTransformer(PCA_COMPONENTS, seed + 2).fit(
        mut_df.loc[train_cells].to_numpy(dtype=float)
    )

    payload = {
        "Xr": rna_tr.transform(rna_df.loc[eligible_cells].to_numpy(dtype=float)),
        "Xc": cnv_tr.transform(cnv_df.loc[eligible_cells].to_numpy(dtype=float)),
        "Xm": mut_tr.transform(mut_df.loc[eligible_cells].to_numpy(dtype=float)),
    }
    dump(payload, path)
    return payload


In [8]:
# Fold-safe standardisation for the target block
class StandardiseObservedBlock:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self.keep_ = None

    def fit(self, X: np.ndarray) -> "StandardiseObservedBlock":
        X = np.asarray(X, dtype=float)
        self.keep_ = np.isfinite(X).any(axis=0)
        if self.keep_ is None or self.keep_.sum() == 0:
            self.mean_ = np.zeros(0, dtype=np.float32)
            self.std_ = np.ones(0, dtype=np.float32)
            return self
        Xk = X[:, self.keep_]
        col_mean = np.nanmean(Xk, axis=0)
        col_std = np.nanstd(Xk, axis=0)
        col_std = np.where(col_std <= 1e-8, 1.0, col_std)
        self.mean_ = col_mean.astype(np.float32)
        self.std_ = col_std.astype(np.float32)
        return self

    def transform(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        X = np.asarray(X, dtype=float)
        if self.keep_ is None or self.keep_.sum() == 0:
            return (
                np.zeros((X.shape[0], 0), dtype=np.float32),
                np.zeros((X.shape[0], 0), dtype=bool),
            )
        Xk = X[:, self.keep_]
        miss = ~np.isfinite(Xk)
        Xfill = np.where(miss, self.mean_[None, :], Xk)
        Xstd = (Xfill - self.mean_[None, :]) / self.std_[None, :]
        return Xstd.astype(np.float32), miss


In [9]:
# Patch helpers
def pad_feature_matrix(X: np.ndarray, multiple: int) -> Tuple[np.ndarray, int]:
    X = np.asarray(X, dtype=np.float32)
    n_rows, n_cols = X.shape
    if n_cols == 0:
        return X, 0
    pad = (multiple - (n_cols % multiple)) % multiple
    if pad == 0:
        return X, 0
    Xpad = np.pad(X, ((0, 0), (0, pad)), mode="constant", constant_values=0.0)
    return Xpad, pad

def pad_mask_matrix(M: np.ndarray, multiple: int) -> Tuple[np.ndarray, int]:
    M = np.asarray(M, dtype=bool)
    n_rows, n_cols = M.shape
    if n_cols == 0:
        return M, 0
    pad = (multiple - (n_cols % multiple)) % multiple
    if pad == 0:
        return M, 0
    Mpad = np.pad(M, ((0, 0), (0, pad)), mode="constant", constant_values=True)
    return Mpad, pad

def make_patch_view(X: np.ndarray, patch_size: int) -> np.ndarray:
    n_rows, n_cols = X.shape
    if n_cols == 0:
        return np.zeros((n_rows, 0, patch_size), dtype=np.float32)
    assert n_cols % patch_size == 0
    return X.reshape(n_rows, n_cols // patch_size, patch_size)

def make_patch_bool_view(M: np.ndarray, patch_size: int) -> np.ndarray:
    n_rows, n_cols = M.shape
    if n_cols == 0:
        return np.zeros((n_rows, 0, patch_size), dtype=bool)
    assert n_cols % patch_size == 0
    return M.reshape(n_rows, n_cols // patch_size, patch_size)

def make_random_remask_patch_mask(
    obs_patch_mask: np.ndarray,
    remask_ratio: float,
    rng: np.random.Generator,
) -> np.ndarray:
    obs_patch_mask = np.asarray(obs_patch_mask, dtype=bool)
    B, P = obs_patch_mask.shape
    out = np.zeros((B, P), dtype=bool)
    for i in range(B):
        idx = np.flatnonzero(obs_patch_mask[i])
        n_obs = len(idx)
        if n_obs <= 1:
            continue
        n_mask = max(1, int(round(n_obs * remask_ratio)))
        n_mask = min(n_mask, n_obs - 1)
        chosen = rng.choice(idx, size=n_mask, replace=False)
        out[i, chosen] = True
    return out

def _split_train_val_idx(n_samples: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    idx = np.arange(n_samples)
    if n_samples < (MIN_VAL_SAMPLES * 2):
        return idx, np.array([], dtype=int)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_val = max(MIN_VAL_SAMPLES, int(round(n_samples * VAL_FRAC)))
    n_val = min(n_val, n_samples // 3)
    val_idx = np.sort(idx[:n_val])
    tr_idx = np.sort(idx[n_val:])
    return tr_idx, val_idx


In [10]:
# ReMasker-style patchified masked autoencoder
class PatchReMasker(nn.Module):
    def __init__(
        self,
        patch_size: int,
        n_patches: int,
        embed_dim: int,
        depth: int,
        num_heads: int,
        ff_mult: float,
        dropout: float,
        aux_dim: int = 0,
    ):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = n_patches
        self.aux_dim = aux_dim

        self.patch_embed = nn.Linear(patch_size * 2, embed_dim)
        self.patch_pos = nn.Parameter(torch.zeros(1, n_patches, embed_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * ff_mult),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)

        dec_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * ff_mult),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerEncoder(dec_layer, num_layers=max(1, depth // 2))
        self.decoder_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, patch_size)

        if aux_dim > 0:
            self.aux_proj = nn.Sequential(
                nn.Linear(aux_dim, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, embed_dim),
            )
            self.aux_pos = nn.Parameter(torch.zeros(1, 1, embed_dim))
        else:
            self.aux_proj = None
            self.aux_pos = None

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        nn.init.normal_(self.patch_pos, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)
        if self.aux_pos is not None:
            nn.init.normal_(self.aux_pos, std=0.02)

    def forward(
        self,
        patch_values: torch.Tensor,
        patch_observed: torch.Tensor,
        remask_patch: torch.Tensor,
        aux_vec: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        x_in = torch.cat([patch_values, patch_observed], dim=-1)
        tok = self.patch_embed(x_in) + self.patch_pos

        mask_tok = self.mask_token.expand(tok.shape[0], tok.shape[1], tok.shape[2])
        tok = torch.where(remask_patch.unsqueeze(-1), mask_tok, tok)

        if self.aux_proj is not None and aux_vec is not None:
            aux_tok = self.aux_proj(aux_vec).unsqueeze(1) + self.aux_pos
            tok = torch.cat([tok, aux_tok], dim=1)
            z = self.encoder(tok)
            z_patch = z[:, :self.n_patches, :]
        else:
            z = self.encoder(tok)
            z_patch = z

        z_patch = self.decoder(z_patch)
        z_patch = self.decoder_norm(z_patch)
        pred = self.head(z_patch)
        return pred


In [11]:
def _train_patch_remasker(
    patch_values_train: np.ndarray,
    patch_observed_train: np.ndarray,
    seed: int,
    aux_train: Optional[np.ndarray] = None,
) -> PatchReMasker:
    B, P, patch_size = patch_values_train.shape
    aux_dim = 0 if aux_train is None else aux_train.shape[1]

    model = PatchReMasker(
        patch_size=patch_size,
        n_patches=P,
        embed_dim=REMASKER_EMBED_DIM,
        depth=REMASKER_DEPTH,
        num_heads=REMASKER_NUM_HEADS,
        ff_mult=REMASKER_FF_MULT,
        dropout=REMASKER_DROPOUT,
        aux_dim=aux_dim,
    ).to(TORCH_DEVICE)

    tr_idx, val_idx = _split_train_val_idx(B, seed)

    x_tr = torch.from_numpy(patch_values_train[tr_idx])
    m_tr = torch.from_numpy(patch_observed_train[tr_idx].astype(np.float32))
    if aux_train is not None:
        a_tr = torch.from_numpy(aux_train[tr_idx].astype(np.float32))
        ds = TensorDataset(x_tr, m_tr, a_tr)
    else:
        ds = TensorDataset(x_tr, m_tr)

    dl = DataLoader(ds, batch_size=min(REMASKER_BATCH_SIZE, len(ds)), shuffle=True)

    if len(val_idx) > 0:
        x_val = patch_values_train[val_idx]
        m_val = patch_observed_train[val_idx]
        a_val = None if aux_train is None else aux_train[val_idx]
    else:
        x_val = None
        m_val = None
        a_val = None

    opt = torch.optim.Adam(
        model.parameters(),
        lr=REMASKER_LR,
        weight_decay=REMASKER_WEIGHT_DECAY,
    )

    best_state = copy.deepcopy(model.state_dict())
    best_loss = math.inf
    bad_epochs = 0
    epoch_rng = np.random.default_rng(seed)

    for _epoch in range(REMASKER_EPOCHS):
        model.train()
        for batch in dl:
            if aux_train is not None:
                xb, mb, ab = batch
                ab = ab.to(TORCH_DEVICE)
            else:
                xb, mb = batch
                ab = None

            xb = xb.to(TORCH_DEVICE)
            mb = mb.to(TORCH_DEVICE)

            obs_patch = mb.bool().any(dim=2).cpu().numpy()
            remask_np = make_random_remask_patch_mask(
                obs_patch_mask=obs_patch,
                remask_ratio=REMASK_RATIO,
                rng=epoch_rng,
            )
            remask = torch.from_numpy(remask_np).to(TORCH_DEVICE)

            pred = model(
                patch_values=xb,
                patch_observed=mb,
                remask_patch=remask,
                aux_vec=ab,
            )

            loss_mask = remask.unsqueeze(-1) & mb.bool()
            denom = loss_mask.float().sum().clamp_min(1.0)
            loss = (((pred - xb) ** 2) * loss_mask.float()).sum() / denom

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if x_val is None:
                current = float(loss.item())
            else:
                obs_patch_val = m_val.any(axis=2)
                remask_val_np = make_random_remask_patch_mask(
                    obs_patch_mask=obs_patch_val,
                    remask_ratio=REMASK_RATIO,
                    rng=np.random.default_rng(seed + 999),
                )
                xv = torch.from_numpy(x_val).to(TORCH_DEVICE)
                mv = torch.from_numpy(m_val.astype(np.float32)).to(TORCH_DEVICE)
                rv = torch.from_numpy(remask_val_np).to(TORCH_DEVICE)
                av = None if a_val is None else torch.from_numpy(a_val.astype(np.float32)).to(TORCH_DEVICE)
                pred_val = model(
                    patch_values=xv,
                    patch_observed=mv,
                    remask_patch=rv,
                    aux_vec=av,
                )
                loss_mask_val = rv.unsqueeze(-1) & mv.bool()
                denom = loss_mask_val.float().sum().clamp_min(1.0)
                val_loss = (((pred_val - xv) ** 2) * loss_mask_val.float()).sum() / denom
                current = float(val_loss.item())

        if current < (best_loss - 1e-6):
            best_loss = current
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= REMASKER_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model


In [12]:
def remasker_impute_block(
    X_train: np.ndarray,
    X_all: np.ndarray,
    random_state: int,
    add_indicators: bool,
    force_indicators: bool,
    n_pca_components: int,
    aux_train: Optional[np.ndarray] = None,
    aux_all: Optional[np.ndarray] = None,
) -> np.ndarray:
    scaler = StandardiseObservedBlock().fit(X_train)
    Xtr_std, miss_tr = scaler.transform(X_train)
    Xal_std, miss_al = scaler.transform(X_all)

    if Xtr_std.shape[1] == 0:
        return np.zeros((X_all.shape[0], 0), dtype=np.float32)

    Xtr_pad, pad = pad_feature_matrix(Xtr_std, PATCH_SIZE)
    Xal_pad, _ = pad_feature_matrix(Xal_std, PATCH_SIZE)

    miss_tr_pad, _ = pad_mask_matrix(miss_tr, PATCH_SIZE)
    miss_al_pad, _ = pad_mask_matrix(miss_al, PATCH_SIZE)

    patch_tr = make_patch_view(Xtr_pad, PATCH_SIZE)
    patch_al = make_patch_view(Xal_pad, PATCH_SIZE)

    obs_tr = (~make_patch_bool_view(miss_tr_pad, PATCH_SIZE)).astype(np.float32)
    obs_al = (~make_patch_bool_view(miss_al_pad, PATCH_SIZE)).astype(np.float32)

    model = _train_patch_remasker(
        patch_values_train=patch_tr,
        patch_observed_train=obs_tr,
        seed=random_state,
        aux_train=aux_train,
    )

    with torch.no_grad():
        pred_al = model(
            patch_values=torch.from_numpy(patch_al).to(TORCH_DEVICE),
            patch_observed=torch.from_numpy(obs_al).to(TORCH_DEVICE),
            remask_patch=torch.zeros((patch_al.shape[0], patch_al.shape[1]), dtype=torch.bool, device=TORCH_DEVICE),
            aux_vec=None if aux_all is None else torch.from_numpy(aux_all.astype(np.float32)).to(TORCH_DEVICE),
        ).cpu().numpy().astype(np.float32)

        pred_tr = model(
            patch_values=torch.from_numpy(patch_tr).to(TORCH_DEVICE),
            patch_observed=torch.from_numpy(obs_tr).to(TORCH_DEVICE),
            remask_patch=torch.zeros((patch_tr.shape[0], patch_tr.shape[1]), dtype=torch.bool, device=TORCH_DEVICE),
            aux_vec=None if aux_train is None else torch.from_numpy(aux_train.astype(np.float32)).to(TORCH_DEVICE),
        ).cpu().numpy().astype(np.float32)

    pred_al_2d = pred_al.reshape(pred_al.shape[0], -1)
    pred_tr_2d = pred_tr.reshape(pred_tr.shape[0], -1)

    if pad > 0:
        pred_al_2d = pred_al_2d[:, :-pad]
        pred_tr_2d = pred_tr_2d[:, :-pad]

    Xal_imp = Xal_std.copy()
    Xtr_imp = Xtr_std.copy()
    Xal_imp[miss_al] = pred_al_2d[miss_al]
    Xtr_imp[miss_tr] = pred_tr_2d[miss_tr]

    if add_indicators or force_indicators:
        ind_tr = miss_tr.astype(np.float32)
        ind_al = miss_al.astype(np.float32)
        ind_any = ind_tr.any(axis=0)
        ind_tr = ind_tr[:, ind_any]
        ind_al = ind_al[:, ind_any]
        if ind_tr.shape[1] > 0:
            ind_sc = StandardScaler()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                Xtr_imp = np.concatenate(
                    [Xtr_imp, ind_sc.fit_transform(ind_tr).astype(np.float32)],
                    axis=1,
                )
                Xal_imp = np.concatenate(
                    [Xal_imp, ind_sc.transform(ind_al).astype(np.float32)],
                    axis=1,
                )

    n, d = Xtr_imp.shape
    n_comp = min(n_pca_components, max(1, n - 1), d)
    pca = CPU_PCA(n_components=n_comp, random_state=random_state)
    pca.fit(Xtr_imp)
    return pca.transform(Xal_imp).astype(np.float32)


In [13]:
# Cache helpers
def _prot_remasker_cache_path(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    force_indicators: bool,
) -> Path:
    return OUT_CACHE / (
        f"prot_remasker__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}"
        f"__strat{_safe_name(strat_name)}__forceind{int(force_indicators)}.joblib"
    )

def _prot_remasker_aux_cache_path(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    force_indicators: bool,
    aux_name: str,
) -> Path:
    return OUT_CACHE / (
        f"prot_remasker_aux__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}"
        f"__strat{_safe_name(strat_name)}__forceind{int(force_indicators)}__aux{_safe_name(aux_name)}.joblib"
    )

def load_or_build_prot_remasker_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    add_indicators: bool,
    force_indicators: bool,
    prot_elig_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _prot_remasker_cache_path(seed, arm, fold_i, strat_name, force_indicators)
    if path.exists():
        return load(path)

    Xp = remasker_impute_block(
        X_train=prot_elig_values[idx_train_full],
        X_all=prot_elig_values,
        random_state=seed + 300,
        add_indicators=add_indicators,
        force_indicators=force_indicators,
        n_pca_components=PCA_COMPONENTS,
        aux_train=None,
        aux_all=None,
    )
    dump(Xp, path)
    return Xp

def load_or_build_prot_remasker_aux_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    aux_name: str,
    add_indicators: bool,
    force_indicators: bool,
    prot_elig_values: np.ndarray,
    aux_all_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _prot_remasker_aux_cache_path(
        seed=seed,
        arm=arm,
        fold_i=fold_i,
        strat_name=strat_name,
        force_indicators=force_indicators,
        aux_name=aux_name,
    )
    if path.exists():
        return load(path)

    Xp = remasker_impute_block(
        X_train=prot_elig_values[idx_train_full],
        X_all=prot_elig_values,
        random_state=seed + 330,
        add_indicators=add_indicators,
        force_indicators=force_indicators,
        n_pca_components=PCA_COMPONENTS,
        aux_train=aux_all_values[idx_train_full],
        aux_all=aux_all_values,
    )
    dump(Xp, path)
    return Xp


In [14]:
# Data load
cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()
group_col = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string").fillna("Unknown").astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print(f"\nCore cohort (RNA ∩ CNV ∩ MUT): {len(core_cells)} cell lines")
print(f"Group column for CV: {group_col}")

prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()

prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][
    ["depmap_id", "compound_id", "y"]
].copy()

prot_arm_data: Dict[str, pd.DataFrame] = {}
for arm in ACTIVE_ARMS:
    p = IN_T2 / f"prot_optional__{arm}.parquet"
    if p.exists():
        prot_arm_data[arm] = normalise_str_index(pd.read_parquet(p))
        print(f"Loaded {arm}: {prot_arm_data[arm].shape}")
    else:
        print(f"[WARN] {arm} not found at {p}, skipping.")

drug_cov = (
    prism_auc.groupby("compound_id")["depmap_id"]
    .nunique().sort_values(ascending=False)
)
selected_drugs = drug_cov.head(N_DRUGS_BAKEOFF).index.tolist()
prism_sel = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}
print(f"\nSelected {len(selected_drugs)} drugs for ReMasker bake-off by PRISM AUC coverage.")



Core cohort (RNA ∩ CNV ∩ MUT): 1079 cell lines
Group column for CV: lineage_1
Loaded prot_combined_union: (1079, 18751)
Loaded prot_procan_depmapSanger: (1079, 7906)
Loaded prot_rppa_ccle: (1079, 144)
Loaded prot_ms_ccle_gygi: (1079, 11780)

Selected 100 drugs for ReMasker bake-off by PRISM AUC coverage.


In [15]:
# Missingness report
print("MISSINGNESS REPORT")

avail_rows = {"rna": {}, "cnv": {}, "mut": {}}
for c in core_cells:
    avail_rows["rna"][c] = 1 if c in rna.index else 0
    avail_rows["cnv"][c] = 1 if c in cnv.index else 0
    avail_rows["mut"][c] = 1 if c in mut.index else 0
for arm, df in prot_arm_data.items():
    avail_rows[arm] = df.reindex(core_cells).notna().any(axis=1).astype(int).to_dict()

availability = pd.DataFrame(avail_rows, index=core_cells)
availability.index.name = "depmap_id"
availability.to_csv(OUT_REPORTS / "modality_availability_matrix.csv")
print(f"Availability matrix: {OUT_REPORTS / 'modality_availability_matrix.csv'}  shape={availability.shape}")

avail_summary = (
    availability.sum().rename("n_cells_present").to_frame()
    .assign(
        n_cells_total=len(core_cells),
        pct_present=lambda d: (d["n_cells_present"] / len(core_cells) * 100).round(2),
    )
)
avail_summary.to_csv(OUT_REPORTS / "modality_availability_summary.csv")
print("\nModality availability summary:")
display(avail_summary)

feat_miss_rows = []
for arm, df in prot_arm_data.items():
    df_core = df.reindex(core_cells)
    col_miss = df_core.isna().mean()
    row_miss = df_core.isna().mean(axis=1)
    feat_miss_rows.append({
        "arm": arm,
        "n_features": int(df_core.shape[1]),
        "n_cells_in_core": int(df_core.shape[0]),
        "overall_missing_pct": float(df_core.isna().mean().mean() * 100),
        "col_miss_q10": float(col_miss.quantile(0.10)),
        "col_miss_q25": float(col_miss.quantile(0.25)),
        "col_miss_q50": float(col_miss.quantile(0.50)),
        "col_miss_q75": float(col_miss.quantile(0.75)),
        "col_miss_q90": float(col_miss.quantile(0.90)),
        "col_miss_q99": float(col_miss.quantile(0.99)),
        "row_miss_q10": float(row_miss.quantile(0.10)),
        "row_miss_q50": float(row_miss.quantile(0.50)),
        "row_miss_q90": float(row_miss.quantile(0.90)),
        "n_cols_fully_missing": int((col_miss == 1.0).sum()),
        "n_rows_fully_missing": int((row_miss == 1.0).sum()),
        "n_cols_zero_missing": int((col_miss == 0.0).sum()),
    })
feat_miss_df = pd.DataFrame(feat_miss_rows)
feat_miss_df.to_csv(OUT_REPORTS / "per_arm_feature_missingness.csv", index=False)
print("\nPer-arm feature missingness:")
display(feat_miss_df)

pat_counts = None
platform_miss_df = None
if TRACK2_ARM in prot_arm_data:
    union_df = prot_arm_data[TRACK2_ARM].reindex(core_cells)
    prefixes = {"ms": "ms__", "rppa": "rppa__", "procan": "procan__"}
    plat_pres = pd.DataFrame(index=union_df.index)
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        plat_pres[key] = union_df[cols].notna().any(axis=1).astype(int) if cols else 0

    def pattern_label(row) -> str:
        parts = [k for k in ["ms", "rppa", "procan"] if row.get(k, 0) == 1]
        return "+".join(parts) if parts else "none"

    pat_counts = (
        plat_pres.apply(pattern_label, axis=1).rename("pattern")
        .value_counts().rename_axis("pattern").reset_index(name="n_cells")
    )
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])
    pat_counts.to_csv(OUT_REPORTS / "combined_union_platform_patterns.csv", index=False)
    print("\nCombined union platform patterns:")
    display(pat_counts)

    pm_rows = []
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        if not cols:
            continue
        n_absent = int(plat_pres[key].eq(0).sum())
        miss_from_abs = n_absent * len(cols)
        miss_total = int(union_df[cols].isna().sum().sum())
        pm_rows.append({
            "platform": key,
            "n_features": len(cols),
            "n_cells_present": int(plat_pres[key].sum()),
            "n_cells_absent": n_absent,
            "frac_cells_present": float(plat_pres[key].mean()),
            "pct_missingness_from_platform_absence": (
                float(miss_from_abs / miss_total * 100) if miss_total > 0 else np.nan
            ),
        })
    platform_miss_df = pd.DataFrame(pm_rows)
    platform_miss_df.to_csv(
        OUT_REPORTS / "combined_union_platform_missingness_contrib.csv",
        index=False,
    )
    print("\nCombined union, missingness from platform absence:")
    display(platform_miss_df)

def build_missingness_report(
    avail_summary: pd.DataFrame,
    feat_miss_df: pd.DataFrame,
    pat_counts: Optional[pd.DataFrame],
    platform_miss_df: Optional[pd.DataFrame],
    track2_arm: str,
    all_seeds: List[int],
    ts: str,
) -> dict:
    report = {
        "generated_at": ts,
        "seeds": all_seeds,
        "active_arms": ACTIVE_ARMS,
        "deprioritised_arm": "prot_rppa_ccle",
        "leakage_note": (
            "All preprocessing, imputation, PCA, and downstream models are fitted "
            "inside CV folds only. No global statistics are used."
        ),
        "modality_availability": avail_summary.reset_index().to_dict(orient="records"),
        "per_arm_feature_missingness": feat_miss_df.to_dict(orient="records"),
        "remasker_settings": {
            "patch_size": PATCH_SIZE,
            "remask_ratio": REMASK_RATIO,
            "embed_dim": REMASKER_EMBED_DIM,
            "depth": REMASKER_DEPTH,
            "num_heads": REMASKER_NUM_HEADS,
        },
    }

    if pat_counts is not None:
        report["combined_union_platform_patterns"] = {
            "arm": track2_arm,
            "structural_missingness_warning": (
                "Combined-union missingness is heavily shaped by platform "
                "availability. Missingness indicators remain compulsory for "
                "combined-union comparisons."
            ),
            "patterns": pat_counts.to_dict(orient="records"),
        }

    if platform_miss_df is not None:
        report["combined_union_platform_missingness_contrib"] = (
            platform_miss_df.to_dict(orient="records")
        )

    report["bakeoff_outputs"] = {
        "detail_file": f"remasker_bakeoff_merged_{len(all_seeds)}seeds.csv",
        "summary_file": f"remasker_bakeoff_summary_merged_{len(all_seeds)}seeds.csv",
        "lock_file": REMASKER_LOCK_FILENAME,
    }
    return report

missingness_report = build_missingness_report(
    avail_summary=avail_summary,
    feat_miss_df=feat_miss_df,
    pat_counts=pat_counts,
    platform_miss_df=platform_miss_df,
    track2_arm=TRACK2_ARM,
    all_seeds=ALL_SEEDS,
    ts=datetime.now(timezone.utc).isoformat(),
)
write_json(missingness_report, OUT_REPORTS / "missingness_report.json")
print(f"\nMissingness report written: {OUT_REPORTS / 'missingness_report.json'}")


MISSINGNESS REPORT
Availability matrix: artifacts/reports/notebook 6_remasker_prot/modality_availability_matrix.csv  shape=(1079, 7)

Modality availability summary:


,n_cells_present,n_cells_total,pct_present
rna,1079,1079,100.00
cnv,1079,1079,100.00
mut,1079,1079,100.00
prot_combined_union,679,1079,62.93
prot_procan_depmapSanger,485,1079,44.95
prot_rppa_ccle,612,1079,56.72
prot_ms_ccle_gygi,304,1079,28.17



Per-arm feature missingness:


,arm,n_features,n_cells_in_core,overall_missing_pct,col_miss_q10,col_miss_q25,col_miss_q50,col_miss_q75,col_miss_q90,col_miss_q99,row_miss_q10,row_miss_q50,row_miss_q90,n_cols_fully_missing,n_rows_fully_missing,n_cols_zero_missing
0,prot_combined_union,18751,1079,74.118180,0.556070,0.718258,0.718258,0.825765,0.915663,0.961075,0.249363,0.760706,1.0,0,400,0
1,prot_procan_depmapSanger,7906,1079,70.717562,0.550510,0.560704,0.668211,0.843373,0.931418,0.975904,0.308981,1.000000,1.0,0,594,0
2,prot_rppa_ccle,144,1079,43.280816,0.432808,0.432808,0.432808,0.432808,0.432808,0.432808,0.000000,0.000000,1.0,0,467,0
3,prot_ms_ccle_gygi,11780,1079,78.876396,0.718258,0.718258,0.732159,0.861214,0.949027,0.979611,0.237267,1.000000,1.0,0,775,0



Combined union platform patterns:


,pattern,n_cells,frac_cells
0,none,400,0.370714
1,ms+rppa+procan,233,0.215941
2,rppa+procan,187,0.173309
3,rppa,128,0.118628
4,ms+rppa,64,0.059314
5,procan,60,0.055607
6,ms+procan,5,0.004634
7,ms,2,0.001854



Combined union, missingness from platform absence:


,platform,n_features,n_cells_present,n_cells_absent,frac_cells_present,pct_missingness_from_platform_absence
0,ms,10847,304,775,0.281742,92.892390
1,rppa,144,612,467,0.567192,100.000000
2,procan,7760,485,594,0.449490,78.405864



Missingness report written: artifacts/reports/notebook 6_remasker_prot/missingness_report.json


In [16]:
# ReMasker strategies
REMASKER_STRATEGIES = [
    ("remasker", False),
    ("remasker+indicators", True),
]


In [17]:
print("REMASKER-STYLE PROTEOMICS BAKE-OFF (3 seeds, leakage-safe)")

all_bakeoff_rows: List[dict] = []
seeds_to_run: List[int] = []
REQUIRED_NEW_COLS = {"config_rank", "model", "feature_set", "uses_prot"}

for run_seed in ALL_SEEDS:
    seed_path = OUT_REPORTS / f"remasker_bakeoff_seed{run_seed}.csv"
    if seed_path.exists():
        existing = pd.read_csv(seed_path)
        if (
            (not REQUIRED_NEW_COLS.issubset(existing.columns))
            or (not checkpoint_matches_expected_grid(existing, EXPECTED_CONFIG_KEYS))
        ):
            print(f"[resume] Seed {run_seed} file exists but is from an old schema/grid, rerunning.")
            seeds_to_run.append(run_seed)
            continue

        existing["seed"] = existing["seed"].astype(int)
        n_drugs_in_file = existing["compound_id"].nunique()
        if n_drugs_in_file >= N_DRUGS_BAKEOFF:
            print(f"[resume] Seed {run_seed} complete ({n_drugs_in_file} drugs), loading.")
            all_bakeoff_rows.extend(existing.to_dict(orient="records"))
        else:
            print(f"[resume] Seed {run_seed} incomplete ({n_drugs_in_file}/{N_DRUGS_BAKEOFF} drugs), will rerun.")
            seeds_to_run.append(run_seed)
    else:
        seeds_to_run.append(run_seed)

if not seeds_to_run:
    print("[resume] All seeds complete, skipping ReMasker loop.")
else:
    print(f"[resume] Seeds remaining: {seeds_to_run}")

for run_seed in seeds_to_run:
    set_seeds(run_seed)
    print(f"  Seed {run_seed}")

    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            print(f"  [SKIP] {arm} not loaded.")
            continue

        arm_cfgs = CONFIGS_BY_ARM.get(arm, [])
        if not arm_cfgs:
            print(f"  [SKIP] {arm} has no configs assigned.")
            continue

        prot_df = prot_arm_data[arm].copy()
        prot_df.index = prot_df.index.astype(str).str.strip()
        prot_core = prot_df.reindex(core_cells)

        has_prot = prot_core.notna().any(axis=1)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] {arm}: only {len(eligible_cells)} eligible cells.")
            continue

        arm_ckpt_path = OUT_REPORTS / f"remasker_bakeoff_seed{run_seed}_{arm}.csv"
        already_done_drugs: set = set()

        if arm_ckpt_path.exists():
            arm_existing = pd.read_csv(arm_ckpt_path)
            if (
                REQUIRED_NEW_COLS.issubset(arm_existing.columns)
                and checkpoint_matches_expected_grid(arm_existing, EXPECTED_CONFIG_KEYS_BY_ARM[arm])
            ):
                arm_existing["seed"] = arm_existing["seed"].astype(int)
                n_drugs_in_arm = arm_existing["compound_id"].nunique()
                if n_drugs_in_arm >= N_DRUGS_BAKEOFF:
                    print(f"  [resume] seed={run_seed} arm={arm} complete ({n_drugs_in_arm} drugs), loading.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    continue
                else:
                    print(f"  [resume] seed={run_seed} arm={arm} partial ({n_drugs_in_arm}/{N_DRUGS_BAKEOFF} drugs), resuming.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    already_done_drugs = set(arm_existing["compound_id"].unique().tolist())
            else:
                print(f"  [resume] seed={run_seed} arm={arm} checkpoint is old schema/grid, rerunning arm.")

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(
            eligible_cells,
            arm_groups,
            N_SPLITS_DESIRED,
            seed=run_seed,
        )
        print(f"  {arm}: {split_name}, {len(eligible_cells)} cells")

        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        prot_elig = prot_core.loc[eligible_cells]
        prot_elig_values = prot_elig.to_numpy(dtype=float)
        fold_cache: Dict[int, dict] = {}
        force_indicators = (arm == "prot_combined_union")

        print(f"    Building/loading ReMasker fold caches for {arm}...")
        for fold_i, (train_idx, _) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]

            base_payload = load_or_build_base_fold_cache(
                seed=run_seed,
                arm=arm,
                fold_i=fold_i,
                train_cells=train_cells,
                eligible_cells=eligible_cells,
                rna_df=rna,
                cnv_df=cnv,
                mut_df=mut,
            )

            X_aux_all = np.concatenate(
                [base_payload["Xr"], base_payload["Xc"], base_payload["Xm"]],
                axis=1,
            )

            fold_entry = {
                "base_mats": {
                    "rna": base_payload["Xr"],
                    "cnv": base_payload["Xc"],
                    "mut": base_payload["Xm"],
                },
                "prot": {},
                "prot_aux": {},
            }

            for strat_name, add_ind in REMASKER_STRATEGIES:
                try:
                    Xp = load_or_build_prot_remasker_fold_cache(
                        seed=run_seed,
                        arm=arm,
                        fold_i=fold_i,
                        strat_name=strat_name,
                        add_indicators=add_ind,
                        force_indicators=force_indicators,
                        prot_elig_values=prot_elig_values,
                        idx_train_full=np.asarray(train_idx, dtype=int),
                    )
                except Exception as e:
                    print(f"    [WARN] ReMasker cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                    Xp = None

                fold_entry["prot"][(strat_name, add_ind, force_indicators)] = Xp

                if USE_AUX_FOR_PROT_IMPUTATION:
                    try:
                        Xp_aux = load_or_build_prot_remasker_aux_fold_cache(
                            seed=run_seed,
                            arm=arm,
                            fold_i=fold_i,
                            strat_name=strat_name,
                            aux_name=AUX_FOR_PROT_NAME,
                            add_indicators=add_ind,
                            force_indicators=force_indicators,
                            prot_elig_values=prot_elig_values,
                            aux_all_values=X_aux_all,
                            idx_train_full=np.asarray(train_idx, dtype=int),
                        )
                    except Exception as e:
                        print(f"    [WARN] auxiliary ReMasker cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                        Xp_aux = None

                    fold_entry["prot_aux"][(strat_name, add_ind, force_indicators)] = Xp_aux

            fold_cache[fold_i] = fold_entry

        n_run = 0
        n_skip = 0

        for drug in selected_drugs:
            if drug in already_done_drugs:
                continue

            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1
                continue

            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            n_run += 1

            if n_run % 5 == 0:
                arm_rows_partial = [
                    r for r in all_bakeoff_rows
                    if r["seed"] == run_seed and r["arm"] == arm
                ]
                if arm_rows_partial:
                    arm_df_partial = pd.DataFrame(arm_rows_partial)
                    arm_df_partial.to_csv(arm_ckpt_path, index=False)
                    cumulative_done = arm_df_partial["compound_id"].nunique()
                    print(f"    [mid-checkpoint] drug {cumulative_done}/{N_DRUGS_BAKEOFF} | {arm_ckpt_path}")

            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, _ in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue

                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]
                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                base_mats = fold_cache[fold_i]["base_mats"]

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)
                    uses_prot = "prot" in cfg_keys

                    X_nonprot = concat_selected_modalities(
                        mats=base_mats,
                        selected_keys=cfg_keys,
                        n_rows=len(eligible_cells),
                    )

                    if not uses_prot:
                        if X_nonprot.shape[1] == 0:
                            continue

                        row = load_or_run_eval_cache(
                            kind="remasker_eval",
                            seed=run_seed,
                            arm=arm,
                            drug=drug,
                            fold_i=fold_i,
                            cfg_rank=cfg_rank,
                            model_name=cfg_model,
                            feature_set=cfg_feature_set,
                            imputer_strategy="no_prot_reference",
                            extra_meta={
                                "uses_prot": False,
                                "add_indicators": False,
                                "force_indicators": False,
                            },
                            X_train=X_nonprot[idx_train],
                            X_test=X_nonprot[idx_test],
                            y_train=y_train,
                            y_test=y_test,
                        )
                        all_bakeoff_rows.append(row)
                        continue

                    cfg_keys_wo_prot = tuple(k for k in cfg_keys if k != "prot")
                    if len(cfg_keys_wo_prot) > 0:
                        X_ref = concat_selected_modalities(
                            mats=base_mats,
                            selected_keys=cfg_keys_wo_prot,
                            n_rows=len(eligible_cells),
                        )
                        if X_ref.shape[1] > 0:
                            row = load_or_run_eval_cache(
                                kind="remasker_eval",
                                seed=run_seed,
                                arm=arm,
                                drug=drug,
                                fold_i=fold_i,
                                cfg_rank=cfg_rank,
                                model_name=cfg_model,
                                feature_set=cfg_feature_set,
                                imputer_strategy="reference_without_prot",
                                extra_meta={
                                    "uses_prot": True,
                                    "add_indicators": False,
                                    "force_indicators": False,
                                },
                                X_train=X_ref[idx_train],
                                X_test=X_ref[idx_test],
                                y_train=y_train,
                                y_test=y_test,
                            )
                            all_bakeoff_rows.append(row)

                    for strat_name, add_ind in REMASKER_STRATEGIES:
                        Xp = fold_cache[fold_i]["prot"].get(
                            (strat_name, add_ind, force_indicators)
                        )
                        if Xp is not None and Xp.shape[1] > 0:
                            parts = []
                            bad_cfg = False
                            for k in cfg_keys:
                                if k == "prot":
                                    parts.append(Xp)
                                else:
                                    arr = base_mats.get(k)
                                    if arr is None or arr.shape[1] == 0:
                                        bad_cfg = True
                                        break
                                    parts.append(arr)

                            if not bad_cfg and len(parts) > 0:
                                Xf = np.concatenate(parts, axis=1)
                                row = load_or_run_eval_cache(
                                    kind="remasker_eval",
                                    seed=run_seed,
                                    arm=arm,
                                    drug=drug,
                                    fold_i=fold_i,
                                    cfg_rank=cfg_rank,
                                    model_name=cfg_model,
                                    feature_set=cfg_feature_set,
                                    imputer_strategy=strat_name,
                                    extra_meta={
                                        "uses_prot": True,
                                        "add_indicators": add_ind,
                                        "force_indicators": force_indicators,
                                    },
                                    X_train=Xf[idx_train],
                                    X_test=Xf[idx_test],
                                    y_train=y_train,
                                    y_test=y_test,
                                )
                                all_bakeoff_rows.append(row)

                        if USE_AUX_FOR_PROT_IMPUTATION:
                            Xp_aux = fold_cache[fold_i]["prot_aux"].get(
                                (strat_name, add_ind, force_indicators)
                            )
                            if Xp_aux is not None and Xp_aux.shape[1] > 0:
                                parts_aux = []
                                bad_cfg_aux = False
                                for k in cfg_keys:
                                    if k == "prot":
                                        parts_aux.append(Xp_aux)
                                    else:
                                        arr = base_mats.get(k)
                                        if arr is None or arr.shape[1] == 0:
                                            bad_cfg_aux = True
                                            break
                                        parts_aux.append(arr)

                                if not bad_cfg_aux and len(parts_aux) > 0:
                                    Xf_aux = np.concatenate(parts_aux, axis=1)
                                    row = load_or_run_eval_cache(
                                        kind="remasker_eval",
                                        seed=run_seed,
                                        arm=arm,
                                        drug=drug,
                                        fold_i=fold_i,
                                        cfg_rank=cfg_rank,
                                        model_name=cfg_model,
                                        feature_set=cfg_feature_set,
                                        imputer_strategy=f"{AUX_FOR_PROT_NAME}::{strat_name}",
                                        extra_meta={
                                            "uses_prot": True,
                                            "add_indicators": add_ind,
                                            "force_indicators": force_indicators,
                                        },
                                        X_train=Xf_aux[idx_train],
                                        X_test=Xf_aux[idx_test],
                                        y_train=y_train,
                                        y_test=y_test,
                                    )
                                    all_bakeoff_rows.append(row)

        print(f"    drugs_run={n_run}, drugs_skipped={n_skip}")

        arm_rows = [
            r for r in all_bakeoff_rows
            if r["seed"] == run_seed and r["arm"] == arm
        ]
        arm_df = pd.DataFrame(arm_rows)
        arm_df.to_csv(arm_ckpt_path, index=False)
        print(f"    [checkpoint] {arm_ckpt_path}  shape={arm_df.shape}")

    seed_df = pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed])
    seed_path = OUT_REPORTS / f"remasker_bakeoff_seed{run_seed}.csv"
    seed_df.to_csv(seed_path, index=False)
    print(f"  Saved seed {run_seed}: {seed_path}  shape={seed_df.shape}")


REMASKER-STYLE PROTEOMICS BAKE-OFF (3 seeds, leakage-safe)
[resume] Seed 19537 complete (100 drugs), loading.
[resume] Seed 1584678 complete (100 drugs), loading.
[resume] Seed 17052356 complete (100 drugs), loading.
[resume] All seeds complete, skipping ReMasker loop.


In [18]:
# Summaries
bakeoff_df = pd.DataFrame(all_bakeoff_rows)
merged_path = OUT_REPORTS / f"remasker_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_df.to_csv(merged_path, index=False)
print(f"\nMerged ReMasker bake-off: {merged_path}  shape={bakeoff_df.shape}")

drug_means = (
    bakeoff_df
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy", "compound_id"],
        as_index=False,
    )
    .agg(
        spearman=("spearman", safe_nanmean),
        r2=("r2", safe_nanmean),
        n_folds=("fold", "nunique"),
    )
)

bakeoff_summary = (
    drug_means
    .groupby(
        ["config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
    )
    .sort_values(["config_rank", "mean_spearman"], ascending=[True, False])
)

base_ref = (
    bakeoff_summary[
        bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ][
        ["config_rank", "arm", "model", "feature_set", "mean_spearman", "imputer_strategy"]
    ]
    .rename(
        columns={
            "mean_spearman": "baseline_mean_spearman",
            "imputer_strategy": "baseline_strategy",
        }
    )
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set"])
)

bakeoff_summary = bakeoff_summary.merge(
    base_ref,
    on=["config_rank", "arm", "model", "feature_set"],
    how="left",
)

bakeoff_summary["delta_vs_baseline"] = np.where(
    bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"]),
    0.0,
    bakeoff_summary["mean_spearman"] - bakeoff_summary["baseline_mean_spearman"],
)

summary_path = OUT_REPORTS / f"remasker_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_summary.to_csv(summary_path, index=False)
print("\nReMasker bake-off summary:")
display(bakeoff_summary)

per_seed_summary = (
    drug_means
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
    )
    .sort_values(["config_rank", "seed", "mean_spearman"], ascending=[True, True, False])
)

per_seed_path = OUT_REPORTS / "remasker_bakeoff_per_seed_summary.csv"
per_seed_summary.to_csv(per_seed_path, index=False)
print("\nPer-seed ReMasker summary:")
display(per_seed_summary)



Merged ReMasker bake-off: artifacts/reports/notebook 6_remasker_prot/remasker_bakeoff_merged_3seeds.csv  shape=(1048800, 15)

ReMasker bake-off summary:


,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_seeds,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,baseline_mean_spearman,baseline_strategy,delta_vs_baseline
0,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,3,100,0.140580,0.149796,0.119701,-0.030691,0.140580,no_prot_reference,0.000000
1,2,prot_combined_union,elasticnet,cnv,False,no_prot_reference,3,100,0.080981,0.067355,0.082828,-0.052511,0.080981,no_prot_reference,0.000000
2,3,prot_combined_union,elasticnet,mut,False,no_prot_reference,3,100,-0.000526,0.005419,0.078003,-0.059956,-0.000526,no_prot_reference,0.000000
3,4,prot_combined_union,elasticnet,prot,True,aux_rna_cnv_mut::remasker,3,100,0.106714,0.106575,0.099647,-0.046246,NaN,NaN,NaN
4,4,prot_combined_union,elasticnet,prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.105806,0.106575,0.099656,-0.046073,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,remasker,3,100,0.138176,0.140068,0.117977,-0.994120,0.092559,reference_without_prot,0.045617
364,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,aux_rna_cnv_mut::remasker,3,100,0.137170,0.140431,0.117166,-1.039384,0.092559,reference_without_prot,0.044611
365,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.124931,0.134463,0.104131,-1.100113,0.092559,reference_without_prot,0.032371
366,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,remasker+indicators,3,100,0.124749,0.132108,0.103338,-1.096385,0.092559,reference_without_prot,0.032190



Per-seed ReMasker summary:


,seed,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_drugs,mean_spearman,median_spearman,std_spearman
0,19537,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.143237,0.149861,0.116273
368,1584678,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.137831,0.144995,0.123819
736,17052356,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.140674,0.155837,0.118826
1,19537,2,prot_combined_union,elasticnet,cnv,False,no_prot_reference,100,0.085342,0.070437,0.079040
369,1584678,2,prot_combined_union,elasticnet,cnv,False,no_prot_reference,100,0.080544,0.064667,0.084023
...,...,...,...,...,...,...,...,...,...,...,...
1099,17052356,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,aux_rna_cnv_mut::remasker,100,0.134867,0.143055,0.114424
1102,17052356,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,remasker,100,0.133266,0.142519,0.117351
1100,17052356,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,aux_rna_cnv_mut::remasker+indicators,100,0.124695,0.133725,0.105044
1103,17052356,120,prot_ms_ccle_gygi,ridge,rna+cnv+mut+prot,True,remasker+indicators,100,0.123154,0.131067,0.101951


In [19]:
# Lock decision
print("REMASKER LOCK DECISION")

remasker_choice = {}

for cfg in REMASKER_TEST_CONFIGS:
    cfg_rank = int(cfg["rank"])
    cfg_arm = str(cfg["arm"])
    cfg_model = str(cfg["model"]).lower()
    cfg_feature_set = str(cfg["feature_set"])
    cfg_uses_prot = "prot" in parse_feature_set(cfg_feature_set)

    key = f"rank{cfg_rank}__{cfg_arm}__{cfg_model}__{cfg_feature_set}"

    cfg_df = bakeoff_summary[
        (bakeoff_summary["config_rank"] == cfg_rank) &
        (bakeoff_summary["arm"] == cfg_arm) &
        (bakeoff_summary["model"] == cfg_model) &
        (bakeoff_summary["feature_set"] == cfg_feature_set)
    ].copy()

    if cfg_df.shape[0] == 0:
        remasker_choice[key] = {
            "chosen_strategy": None,
            "reason": "No ReMasker bake-off rows produced for this config.",
        }
        continue

    if not cfg_uses_prot:
        ref_rows = cfg_df[cfg_df["imputer_strategy"] == "no_prot_reference"]
        ref_sp = float(ref_rows["mean_spearman"].iloc[0]) if len(ref_rows) else np.nan

        remasker_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": "not_applicable",
            "add_indicators": False,
            "force_indicators_track2": False,
            "mean_spearman_chosen": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "mean_spearman_baseline": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "delta_vs_baseline": 0.0,
            "n_seeds_in_estimate": len(ALL_SEEDS),
            "reason": "This ranked configuration does not contain proteomics, so ReMasker imputation is not applicable.",
        }
        print(f"\n{key}:")
        print("  Chosen   : not_applicable")
        continue

    candidates = cfg_df[
        ~cfg_df["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ].copy()

    if cfg_arm == "prot_combined_union":
        ind_candidates = candidates[candidates["imputer_strategy"].str.contains("indicator", regex=False)]
        if ind_candidates.shape[0] > 0:
            candidates = ind_candidates

    if candidates.shape[0] == 0:
        remasker_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": None,
            "reason": "No ReMasker candidates available for this config.",
        }
        continue

    best = candidates.sort_values("mean_spearman", ascending=False).iloc[0]
    base_rows = cfg_df[cfg_df["imputer_strategy"] == "reference_without_prot"]
    base_sp = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan

    chosen = str(best["imputer_strategy"])
    mean_sp = float(best["mean_spearman"])
    std_sp = float(best["std_spearman"])
    delta = float(best["delta_vs_baseline"]) if pd.notna(best["delta_vs_baseline"]) else np.nan

    remasker_choice[key] = {
        "config_rank": cfg_rank,
        "arm": cfg_arm,
        "model": cfg_model,
        "feature_set": cfg_feature_set,
        "chosen_strategy": chosen,
        "add_indicators": ("indicator" in chosen) or (cfg_arm == "prot_combined_union"),
        "force_indicators_track2": (cfg_arm == "prot_combined_union"),
        "mean_spearman_chosen": round(mean_sp, 6),
        "std_spearman_chosen": round(std_sp, 6),
        "mean_spearman_baseline": round(base_sp, 6) if np.isfinite(base_sp) else None,
        "delta_vs_baseline": round(delta, 6) if np.isfinite(delta) else None,
        "n_seeds_in_estimate": len(ALL_SEEDS),
        "reason": (
            f"Best mean Spearman ({mean_sp:.4f} ± {std_sp:.4f}) across {len(ALL_SEEDS)} seeds"
            + (
                f"; delta vs config-specific no-prot reference: {delta:+.4f}."
                if np.isfinite(delta) else
                "; no config-specific no-prot reference available."
            )
            + (" Indicators mandatory for combined union arm." if cfg_arm == "prot_combined_union" else "")
        ),
    }

    print(f"\n{key}:")
    print(f"  Chosen   : {chosen}")
    print(f"  Spearman : {mean_sp:.4f} ± {std_sp:.4f}")
    if np.isfinite(base_sp):
        print(f"  Baseline : {base_sp:.4f}  delta={delta:+.4f}")

remasker_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target": PRIMARY_TARGET,
    "pca_components": PCA_COMPONENTS,
    "patch_size": PATCH_SIZE,
    "remask_ratio": REMASK_RATIO,
    "torch_device": str(TORCH_DEVICE),
    "remasker_settings": {
        "embed_dim": REMASKER_EMBED_DIM,
        "depth": REMASKER_DEPTH,
        "num_heads": REMASKER_NUM_HEADS,
        "ff_mult": REMASKER_FF_MULT,
        "dropout": REMASKER_DROPOUT,
        "epochs": REMASKER_EPOCHS,
        "patience": REMASKER_PATIENCE,
        "batch_size": REMASKER_BATCH_SIZE,
        "lr": REMASKER_LR,
        "weight_decay": REMASKER_WEIGHT_DECAY,
    },
    "test_configs": REMASKER_TEST_CONFIGS,
    "note": (
        "This notebook uses a patchified ReMasker-style masked autoencoder so that "
        "high-dimensional proteomics blocks remain tractable across the full "
        "feature-combination grid. All transforms, ReMasker training, PCA, and "
        "downstream regressors are fit inside training folds only. Auxiliary "
        "ReMasker uses fold-safe RNA, CNV, and mutation embeddings from the same "
        "fold only."
    ),
    "config_choices": remasker_choice,
}

remasker_choice_path = OUT_META / REMASKER_LOCK_FILENAME
write_json(remasker_choice_doc, remasker_choice_path)
print(f"\nReMasker imputer choice written: {remasker_choice_path}")

global_copy = IN_META / REMASKER_LOCK_FILENAME
write_json(remasker_choice_doc, global_copy)
print(f"Global copy written: {global_copy}")


REMASKER LOCK DECISION

rank1__prot_combined_union__elasticnet__rna:
  Chosen   : not_applicable

rank2__prot_combined_union__elasticnet__cnv:
  Chosen   : not_applicable

rank3__prot_combined_union__elasticnet__mut:
  Chosen   : not_applicable

rank4__prot_combined_union__elasticnet__prot:
  Chosen   : aux_rna_cnv_mut::remasker+indicators
  Spearman : 0.1058 ± 0.0997

rank5__prot_combined_union__elasticnet__rna+cnv:
  Chosen   : not_applicable

rank6__prot_combined_union__elasticnet__rna+mut:
  Chosen   : not_applicable

rank7__prot_combined_union__elasticnet__rna+prot:
  Chosen   : aux_rna_cnv_mut::remasker+indicators
  Spearman : 0.1499 ± 0.1117
  Baseline : 0.1406  delta=+0.0094

rank8__prot_combined_union__elasticnet__cnv+mut:
  Chosen   : not_applicable

rank9__prot_combined_union__elasticnet__cnv+prot:
  Chosen   : remasker+indicators
  Spearman : 0.1173 ± 0.0957
  Baseline : 0.0810  delta=+0.0364

rank10__prot_combined_union__elasticnet__mut+prot:
  Chosen   : aux_rna_cnv_mut::

In [ ]:
FIG_DIR = OUT_REPORTS / "figures"
INTERP_DIR = OUT_REPORTS / "interpretability"
FIG_DIR.mkdir(parents=True, exist_ok=True)
INTERP_DIR.mkdir(parents=True, exist_ok=True)

print("Saving evaluation figures to:", FIG_DIR)
print("Saving interpretability outputs to:", INTERP_DIR)


def safe_filename(s: str) -> str:
    return (
        str(s)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace("+", "plus")
        .replace(":", "_")
        .replace("|", "_")
    )

def wrap_label(s: str, width: int = 24) -> str:
    return "\n".join(textwrap.wrap(str(s), width=width))

def short_arm_label(s: str) -> str:
    mapping = {
        "prot_combined_union": "Combined",
        "prot_procan_depmapSanger": "ProCan",
        "prot_rppa_ccle": "RPPA",
        "prot_ms_ccle_gygi": "MS Gygi",
    }
    return mapping.get(str(s), str(s))

def short_imputer_label(s: str) -> str:
    s = str(s)
    mapping = {
        "remasker": "ReMasker",
        "remasker+indicators": "ReMasker+ind",
        "aux_rna_cnv_mut::remasker": "Aux ReMasker",
        "aux_rna_cnv_mut::remasker+indicators": "Aux ReMasker+ind",
        "reference_without_prot": "No-prot ref",
        "no_prot_reference": "No-prot ref",
    }
    return mapping.get(s, s)

def short_compound_label(s: str, max_len: int = 36) -> str:
    s = str(s)
    return s if len(s) <= max_len else s[: max_len - 3] + "..."

def config_label_from_cols(df: pd.DataFrame) -> pd.Series:
    rank = pd.to_numeric(df["config_rank"], errors="coerce").fillna(-1).astype(int).astype(str)
    return (
        "r" + rank
        + " | " + df["arm"].map(short_arm_label).astype(str)
        + " | " + df["model"].astype(str)
        + " | " + df["feature_set"].astype(str)
    )

def safe_mean(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.mean()) if x.size else np.nan

def safe_median(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if x.size else np.nan

def safe_std(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.std()) if x.size else np.nan

def finish_plot(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()

def draw_matrix_heatmap(
    mat: pd.DataFrame,
    title: str,
    out_path: Path,
    centre_zero: bool = False,
    annotate_fmt: str = "{:.3f}",
    extra_annot: Optional[pd.DataFrame] = None,
    max_annot_rows: int = 28,
):
    if mat is None or mat.empty:
        print(f"[skip] {title}: empty matrix")
        return

    arr = mat.to_numpy(dtype=float)
    if not np.isfinite(arr).any():
        print(f"[skip] {title}: no finite values")
        return

    vmin = float(np.nanmin(arr))
    vmax = float(np.nanmax(arr))

    fig_w = max(8.5, 1.45 * max(4, mat.shape[1]) + 2.0)
    fig_h = min(16.0, max(4.8, 0.42 * max(4, mat.shape[0]) + 2.0))

    plt.figure(figsize=(fig_w, fig_h))

    if centre_zero and (vmin < 0.0) and (vmax > 0.0):
        vmax_abs = max(abs(vmin), abs(vmax))
        norm = TwoSlopeNorm(vmin=-vmax_abs, vcenter=0.0, vmax=vmax_abs)
        im = plt.imshow(arr, aspect="auto", norm=norm)
    else:
        im = plt.imshow(arr, aspect="auto")

    plt.colorbar(im, label="Value")
    plt.xticks(
        range(mat.shape[1]),
        [wrap_label(c, 18) for c in mat.columns],
        rotation=35,
        ha="right",
        fontsize=9,
    )
    plt.yticks(
        range(mat.shape[0]),
        [wrap_label(i, 28) for i in mat.index],
        fontsize=8.5 if mat.shape[0] > 16 else 9,
    )
    plt.title(title)

    if mat.shape[0] <= max_annot_rows:
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = arr[i, j]
                if np.isfinite(val):
                    txt = annotate_fmt.format(val)
                    if extra_annot is not None:
                        ev = extra_annot.iloc[i, j]
                        if pd.notna(ev):
                            txt = f"{txt}\n({ev})"
                    plt.text(
                        j,
                        i,
                        txt,
                        ha="center",
                        va="center",
                        fontsize=8,
                        linespacing=0.9,
                    )

    plt.subplots_adjust(left=0.34, bottom=0.20, right=0.96, top=0.92)
    finish_plot(out_path)

def select_top_rows_by_abs_signal(
    mat: pd.DataFrame,
    extra_annot: Optional[pd.DataFrame] = None,
    max_rows: int = 24,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    if mat.shape[0] <= max_rows:
        return mat, extra_annot
    keep = mat.abs().max(axis=1).sort_values(ascending=False).head(max_rows).index.tolist()
    mat_sel = mat.loc[keep]
    ann_sel = extra_annot.loc[keep] if extra_annot is not None else None
    return mat_sel, ann_sel

summary_all = bakeoff_summary.copy()
detail_all = bakeoff_df.copy()

for df_ in [summary_all, detail_all]:
    if df_.shape[0] > 0:
        for c in [
            "seed",
            "config_rank",
            "fold",
            "mean_spearman",
            "median_spearman",
            "std_spearman",
            "mean_r2",
            "delta_vs_baseline",
            "spearman",
            "r2",
            "n_train",
            "n_test",
        ]:
            if c in df_.columns:
                df_[c] = pd.to_numeric(df_[c], errors="coerce")

if "uses_prot" in summary_all.columns:
    summary_all["uses_prot"] = summary_all["uses_prot"].astype(str).str.lower().eq("true")

ref_strategies = {"no_prot_reference", "reference_without_prot"}
aux_prefix = f"{AUX_FOR_PROT_NAME}::" if "AUX_FOR_PROT_NAME" in globals() else "aux_rna_cnv_mut::"

remasker_only_summary = summary_all[
    (summary_all["uses_prot"] == True)
    & (~summary_all["imputer_strategy"].isin(ref_strategies))
].copy()

best_by_config = (
    remasker_only_summary
    .sort_values(
        ["config_rank", "delta_vs_baseline", "mean_spearman", "median_spearman"],
        ascending=[True, False, False, False],
    )
    .groupby(["config_rank", "arm", "model", "feature_set"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# bakeoff_summary already carries baseline_mean_spearman, so avoid re-merging it
if "baseline_mean_spearman" not in best_by_config.columns:
    baseline_cfg = summary_all[
        summary_all["imputer_strategy"] == "reference_without_prot"
    ][["config_rank", "arm", "model", "feature_set", "mean_spearman"]].rename(
        columns={"mean_spearman": "baseline_mean_spearman"}
    )

    best_by_config = best_by_config.merge(
        baseline_cfg,
        on=["config_rank", "arm", "model", "feature_set"],
        how="left",
    )

best_by_config["config_label"] = config_label_from_cols(best_by_config)
best_by_config.to_csv(OUT_REPORTS / "best_remasker_by_config.csv", index=False)

best_by_arm = (
    best_by_config[best_by_config["delta_vs_baseline"].notna()]
    .sort_values(
        ["arm", "delta_vs_baseline", "mean_spearman", "median_spearman"],
        ascending=[True, False, False, False],
    )
    .groupby("arm", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_by_arm.to_csv(OUT_REPORTS / "best_remasker_by_arm.csv", index=False)

display(best_by_config.head(20))
display(best_by_arm)

# Data quality versus utility frontier

if (
    isinstance(feat_miss_df, pd.DataFrame)
    and feat_miss_df.shape[0] > 0
    and best_by_arm.shape[0] > 0
):
    frontier_df = feat_miss_df.merge(
        best_by_arm[
            [
                "arm",
                "imputer_strategy",
                "model",
                "feature_set",
                "mean_spearman",
                "delta_vs_baseline",
                "baseline_mean_spearman",
            ]
        ],
        on="arm",
        how="left",
    )

    frontier_df["arm_short"] = frontier_df["arm"].map(short_arm_label)
    frontier_df.to_csv(OUT_REPORTS / "quality_utility_frontier.csv", index=False)

    plt.figure(figsize=(8.8, 5.8))
    sizes = 80 + 220 * (frontier_df["n_features"] / frontier_df["n_features"].max())
    plt.scatter(
        frontier_df["overall_missing_pct"],
        frontier_df["delta_vs_baseline"],
        s=sizes,
    )
    plt.axhline(0.0, linestyle="--", linewidth=1.0)

    for _, r in frontier_df.iterrows():
        if pd.notna(r["delta_vs_baseline"]):
            plt.text(
                r["overall_missing_pct"],
                r["delta_vs_baseline"],
                f"{r['arm_short']}\nsp={r['mean_spearman']:.3f}",
                fontsize=9,
                ha="left",
                va="bottom",
            )

    plt.title("Proteomics arm quality versus predictive utility")
    plt.xlabel("Overall proteomics missingness (%)")
    plt.ylabel("Best delta vs no-proteomics baseline")
    finish_plot(FIG_DIR / "quality_utility_frontier.png")


# Arm by strategy uplift heatmap
# This directly shows which strategy works for which arm.

if remasker_only_summary.shape[0] > 0:
    arm_strategy = (
        remasker_only_summary
        .groupby(["arm", "imputer_strategy"], as_index=False)
        .agg(
            mean_delta=("delta_vs_baseline", safe_mean),
            mean_spearman=("mean_spearman", safe_mean),
            n_configs=("config_rank", "nunique"),
        )
    )

    arm_strategy["arm_label"] = arm_strategy["arm"].map(short_arm_label)
    arm_strategy["imp_label"] = arm_strategy["imputer_strategy"].map(short_imputer_label)

    hm = arm_strategy.pivot(index="arm_label", columns="imp_label", values="mean_delta")
    ann = arm_strategy.assign(
        lbl=arm_strategy["mean_spearman"].map(lambda x: f"sp={x:.3f}" if pd.notna(x) else "")
    ).pivot(index="arm_label", columns="imp_label", values="lbl").reindex(index=hm.index, columns=hm.columns)

    draw_matrix_heatmap(
        hm,
        title=f"Mean uplift by proteomics arm and ReMasker strategy ({PRIMARY_TARGET})",
        out_path=FIG_DIR / f"arm_strategy_uplift_heatmap_{PRIMARY_TARGET}.png",
        centre_zero=True,
        annotate_fmt="{:+.3f}",
        extra_annot=ann,
    )

    arm_strategy.to_csv(OUT_REPORTS / "arm_strategy_uplift_summary.csv", index=False)


# Baseline versus best-imputed comparison at config level
# This answers whether ReMasker actually moves configs above baseline.

cfg_vs_base = best_by_config[best_by_config["baseline_mean_spearman"].notna()].copy()

if cfg_vs_base.shape[0] > 0:
    cfg_vs_base["gain"] = cfg_vs_base["mean_spearman"] - cfg_vs_base["baseline_mean_spearman"]
    cfg_vs_base.to_csv(OUT_REPORTS / "config_level_baseline_vs_best.csv", index=False)

    plt.figure(figsize=(7.8, 6.6))
    plt.scatter(
        cfg_vs_base["baseline_mean_spearman"],
        cfg_vs_base["mean_spearman"],
    )

    lo = float(
        min(
            cfg_vs_base["baseline_mean_spearman"].min(),
            cfg_vs_base["mean_spearman"].min(),
        )
    )
    hi = float(
        max(
            cfg_vs_base["baseline_mean_spearman"].max(),
            cfg_vs_base["mean_spearman"].max(),
        )
    )
    pad = max(0.01, 0.08 * (hi - lo))
    plt.plot([lo - pad, hi + pad], [lo - pad, hi + pad], linestyle="--", linewidth=1.0)

    annotate_df = cfg_vs_base.reindex(
        cfg_vs_base["gain"].abs().sort_values(ascending=False).head(min(12, cfg_vs_base.shape[0])).index
    )

    for _, r in annotate_df.iterrows():
        plt.text(
            r["baseline_mean_spearman"],
            r["mean_spearman"],
            wrap_label(
                f"{short_arm_label(r['arm'])} | {r['model']} | {r['feature_set']}",
                18,
            ),
            fontsize=8.5,
            ha="left",
            va="bottom",
        )

    plt.title(f"Best ReMasker result versus no-proteomics baseline by config ({PRIMARY_TARGET})")
    plt.xlabel("Baseline mean Spearman")
    plt.ylabel("Best imputed mean Spearman")
    finish_plot(FIG_DIR / f"baseline_vs_best_config_scatter_{PRIMARY_TARGET}.png")


# Seed-stability heatmap for the best config in each arm

if best_by_arm.shape[0] > 0 and isinstance(per_seed_summary, pd.DataFrame) and per_seed_summary.shape[0] > 0:
    per_seed_best = per_seed_summary.merge(
        best_by_arm[
            ["config_rank", "arm", "model", "feature_set", "imputer_strategy"]
        ],
        on=["config_rank", "arm", "model", "feature_set", "imputer_strategy"],
        how="inner",
    )

    per_seed_base = per_seed_summary[
        per_seed_summary["imputer_strategy"] == "reference_without_prot"
    ][["seed", "config_rank", "arm", "model", "feature_set", "mean_spearman"]].rename(
        columns={"mean_spearman": "baseline_mean_spearman"}
    )

    seed_delta = per_seed_best.merge(
        per_seed_base,
        on=["seed", "config_rank", "arm", "model", "feature_set"],
        how="left",
    )
    seed_delta["delta_vs_baseline"] = seed_delta["mean_spearman"] - seed_delta["baseline_mean_spearman"]
    seed_delta["arm_label"] = seed_delta["arm"].map(short_arm_label)
    seed_delta.to_csv(OUT_REPORTS / "seed_stability_best_by_arm.csv", index=False)

    hm = seed_delta.pivot(index="arm_label", columns="seed", values="delta_vs_baseline")
    ann = seed_delta.assign(
        lbl=seed_delta["mean_spearman"].map(lambda x: f"sp={x:.3f}" if pd.notna(x) else "")
    ).pivot(index="arm_label", columns="seed", values="lbl").reindex(index=hm.index, columns=hm.columns)

    draw_matrix_heatmap(
        hm,
        title=f"Seed stability of best arm-specific ReMasker configuration ({PRIMARY_TARGET})",
        out_path=FIG_DIR / f"seed_stability_best_by_arm_{PRIMARY_TARGET}.png",
        centre_zero=True,
        annotate_fmt="{:+.3f}",
        extra_annot=ann,
        max_annot_rows=20,
    )


# Drug-level consistency for the best config in each arm
# This distinguishes broad gains from a few isolated wins.


if detail_all.shape[0] > 0 and best_by_arm.shape[0] > 0:
    drug_level = (
        detail_all
        .groupby(
            ["seed", "config_rank", "arm", "model", "feature_set", "imputer_strategy", "compound_id"],
            as_index=False,
        )
        .agg(
            spearman=("spearman", safe_mean),
            r2=("r2", safe_mean),
        )
    )

    chosen_drug = drug_level.merge(
        best_by_arm[
            ["config_rank", "arm", "model", "feature_set", "imputer_strategy"]
        ],
        on=["config_rank", "arm", "model", "feature_set", "imputer_strategy"],
        how="inner",
    )

    baseline_drug = drug_level[
        drug_level["imputer_strategy"] == "reference_without_prot"
    ][["seed", "config_rank", "arm", "model", "feature_set", "compound_id", "spearman"]].rename(
        columns={"spearman": "baseline_spearman"}
    )

    chosen_drug = chosen_drug.merge(
        baseline_drug,
        on=["seed", "config_rank", "arm", "model", "feature_set", "compound_id"],
        how="left",
    )
    chosen_drug = chosen_drug[chosen_drug["baseline_spearman"].notna()].copy()
    chosen_drug["delta_spearman"] = chosen_drug["spearman"] - chosen_drug["baseline_spearman"]
    chosen_drug["arm_label"] = chosen_drug["arm"].map(short_arm_label)
    chosen_drug.to_csv(OUT_REPORTS / "drug_level_best_by_arm_deltas.csv", index=False)

    arm_consistency = (
        chosen_drug
        .groupby("arm_label", as_index=False)
        .agg(
            mean_delta=("delta_spearman", safe_mean),
            median_delta=("delta_spearman", safe_median),
            win_rate=("delta_spearman", lambda x: float(np.mean(pd.Series(x).dropna() > 0)) if pd.Series(x).dropna().shape[0] > 0 else np.nan),
            n_drug_seed_pairs=("compound_id", "count"),
        )
        .sort_values(["median_delta", "win_rate"], ascending=False)
    )
    arm_consistency.to_csv(OUT_REPORTS / "arm_level_drug_consistency.csv", index=False)
    display(arm_consistency)

    plt.figure(figsize=(8.0, 5.8))
    plt.scatter(
        arm_consistency["win_rate"],
        arm_consistency["median_delta"],
        s=120,
    )
    plt.axhline(0.0, linestyle="--", linewidth=1.0)

    for _, r in arm_consistency.iterrows():
        plt.text(
            r["win_rate"],
            r["median_delta"],
            r["arm_label"],
            fontsize=9,
            ha="left",
            va="bottom",
        )

    plt.title("Drug-level breadth of benefit for the best arm-specific ReMasker config")
    plt.xlabel("Fraction of drug-seed pairs with positive uplift")
    plt.ylabel("Median delta vs no-proteomics baseline")
    finish_plot(FIG_DIR / "drug_level_consistency_by_arm.png")

    drug_arm_mat = (
        chosen_drug
        .groupby(["compound_id", "arm_label"], as_index=False)
        .agg(mean_delta=("delta_spearman", safe_mean))
        .pivot(index="compound_id", columns="arm_label", values="mean_delta")
    )

    if drug_arm_mat is not None and not drug_arm_mat.empty:
        keep = (
            drug_arm_mat.abs().max(axis=1)
            .sort_values(ascending=False)
            .head(min(24, drug_arm_mat.shape[0]))
            .index.tolist()
        )
        drug_arm_mat = drug_arm_mat.loc[keep]
        drug_arm_mat.index = [short_compound_label(x, 34) for x in drug_arm_mat.index]
        drug_arm_mat.to_csv(OUT_REPORTS / "drug_arm_delta_matrix_best_configs.csv")

        draw_matrix_heatmap(
            drug_arm_mat,
            title=f"Drug-level uplift for the best ReMasker config in each arm ({PRIMARY_TARGET})",
            out_path=FIG_DIR / f"drug_arm_delta_heatmap_best_configs_{PRIMARY_TARGET}.png",
            centre_zero=True,
            annotate_fmt="{:+.3f}",
            extra_annot=None,
            max_annot_rows=24,
        )


artefact_index = {
    "remasker_detail_csv": str(merged_path),
    "remasker_summary_csv": str(summary_path),
    "remasker_per_seed_summary_csv": str(per_seed_path),
    "best_remasker_by_config_csv": str(OUT_REPORTS / "best_remasker_by_config.csv"),
    "best_remasker_by_arm_csv": str(OUT_REPORTS / "best_remasker_by_arm.csv"),
    "quality_utility_frontier_csv": str(OUT_REPORTS / "quality_utility_frontier.csv"),
    "config_level_baseline_vs_best_csv": str(OUT_REPORTS / "config_level_baseline_vs_best.csv"),
    "seed_stability_best_by_arm_csv": str(OUT_REPORTS / "seed_stability_best_by_arm.csv"),
    "drug_level_best_by_arm_deltas_csv": str(OUT_REPORTS / "drug_level_best_by_arm_deltas.csv"),
    "arm_level_drug_consistency_csv": str(OUT_REPORTS / "arm_level_drug_consistency.csv"),
    "missingness_report_json": str(OUT_REPORTS / "missingness_report.json"),
    "remasker_choice_json": str(remasker_choice_path),
    "figures_dir": str(FIG_DIR),
    "interpretability_dir": str(INTERP_DIR),
}

write_json(artefact_index, OUT_REPORTS / "artefact_index.json")
display(pd.DataFrame({"artefact": list(artefact_index.keys()), "path": list(artefact_index.values())}))
print("Artefact index written:", OUT_REPORTS / "artefact_index.json")

Saving evaluation figures to: artifacts/reports/notebook 6_remasker_prot/figures
Saving interpretability outputs to: artifacts/reports/notebook 6_remasker_prot/interpretability


,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_seeds,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,baseline_mean_spearman_x,baseline_strategy,delta_vs_baseline,baseline_mean_spearman_y,config_label
0,4,prot_combined_union,elasticnet,prot,True,aux_rna_cnv_mut::remasker,3,100,0.106714,0.106575,0.099647,-0.046246,NaN,NaN,NaN,NaN,r4 | Combined | elasticnet | prot
1,7,prot_combined_union,elasticnet,rna+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.149933,0.154938,0.111699,-0.029071,0.140580,reference_without_prot,0.009353,0.140580,r7 | Combined | elasticnet | rna+prot
2,9,prot_combined_union,elasticnet,cnv+prot,True,remasker,3,100,0.118375,0.105230,0.092164,-0.043287,0.080981,reference_without_prot,0.037394,0.080981,r9 | Combined | elasticnet | cnv+prot
3,10,prot_combined_union,elasticnet,mut+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.093503,0.091702,0.103389,-0.046907,-0.000526,reference_without_prot,0.094029,-0.000526,r10 | Combined | elasticnet | mut+prot
4,12,prot_combined_union,elasticnet,rna+cnv+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.152440,0.156699,0.107316,-0.028682,0.144693,reference_without_prot,0.007747,0.144693,r12 | Combined | elasticnet | rna+cnv+prot
5,13,prot_combined_union,elasticnet,rna+mut+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.139582,0.147796,0.117731,-0.029714,0.130029,reference_without_prot,0.009554,0.130029,r13 | Combined | elasticnet | rna+mut+prot
6,14,prot_combined_union,elasticnet,cnv+mut+prot,True,remasker+indicators,3,100,0.104613,0.095676,0.100552,-0.043440,0.063787,reference_without_prot,0.040826,0.063787,r14 | Combined | elasticnet | cnv+mut+prot
7,15,prot_combined_union,elasticnet,rna+cnv+mut+prot,True,remasker+indicators,3,100,0.143283,0.141297,0.111243,-0.028955,0.134645,reference_without_prot,0.008638,0.134645,r15 | Combined | elasticnet | rna+cnv+mut+prot
8,19,prot_combined_union,ridge,prot,True,aux_rna_cnv_mut::remasker,3,100,0.125278,0.130521,0.090565,-0.203207,NaN,NaN,NaN,NaN,r19 | Combined | ridge | prot
9,22,prot_combined_union,ridge,rna+prot,True,aux_rna_cnv_mut::remasker,3,100,0.098848,0.095033,0.083408,-0.964917,0.176910,reference_without_prot,-0.078061,0.176910,r22 | Combined | ridge | rna+prot


,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_seeds,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,baseline_mean_spearman_x,baseline_strategy,delta_vs_baseline,baseline_mean_spearman_y,config_label
0,10,prot_combined_union,elasticnet,mut+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.093503,0.091702,0.103389,-0.046907,-0.000526,reference_without_prot,0.094029,-0.000526,r10 | Combined | elasticnet | mut+prot
1,100,prot_ms_ccle_gygi,elasticnet,mut+prot,True,aux_rna_cnv_mut::remasker,3,100,0.139234,0.137995,0.126577,-0.107919,0.007854,reference_without_prot,0.131380,0.007854,r100 | MS Gygi | elasticnet | mut+prot
2,40,prot_procan_depmapSanger,elasticnet,mut+prot,True,aux_rna_cnv_mut::remasker+indicators,3,100,0.159990,0.147716,0.135225,-0.036850,0.001534,reference_without_prot,0.158456,0.001534,r40 | ProCan | elasticnet | mut+prot
3,85,prot_rppa_ccle,ridge,mut+prot,True,remasker,3,100,0.072912,0.082918,0.074860,-2.026234,-0.009289,reference_without_prot,0.082202,-0.009289,r85 | RPPA | ridge | mut+prot


KeyError: "['baseline_mean_spearman'] not in index"